In [1]:
"""
UAV Dataset Preparation Pipeline
=================================
Steps:
  1. Load dataset from incident folders
  2. Compute uniqueness & remove near-duplicates via FiftyOne
  3. Split per incident folder into train / val / test
  4. Export final dataset in a clean directory structure

Requirements:
    pip install fiftyone scikit-learn
"""

import os
import shutil
from pathlib import Path
from collections import defaultdict

import fiftyone as fo
import fiftyone.brain as fob
from fiftyone import ViewField as F
from sklearn.model_selection import train_test_split

# ─────────────────────────────────────────────
# CONFIG — edit these before running
# ─────────────────────────────────────────────
DATASET_DIR   = "/home/robby/workspace/Oil-Spill-Benz/datasets/raw/dv5_new"   # Root folder; subfolders = incident classes
OUTPUT_DIR    = "/home/robby/workspace/Oil-Spill-Benz/datasets/processed/dv5_unique"         # Where clean split dataset will be written
DATASET_NAME  = "uav_incidents"

UNIQUENESS_THRESHOLD = 0.75   # 0 = all duplicates kept, 1 = only perfectly unique kept
                              # Start at 0.5; tune after visual inspection

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15            # Must sum to 1.0

RANDOM_SEED = 42
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".tiff", ".tif"}
# ─────────────────────────────────────────────

In [2]:

# ── STEP 1: Load dataset ─────────────────────────────────────────────────────

def load_dataset(dataset_dir: str, name: str) -> fo.Dataset:
    """
    Load images from incident subfolders.
    Each subfolder name becomes the 'incident' tag on every sample.
    """
    print(f"\n{'='*60}")
    print("STEP 1 — Loading dataset")
    print(f"{'='*60}")

    # Delete existing dataset with same name if re-running
    if fo.dataset_exists(name):
        fo.delete_dataset(name)

    dataset = fo.Dataset(name=name, persistent=True)
    samples = []

    root = Path(dataset_dir)
    incident_folders = [f for f in root.iterdir() if f.is_dir()]

    if not incident_folders:
        raise ValueError(f"No subfolders found in {dataset_dir}. "
                         "Each incident should be its own subfolder.")

    for folder in sorted(incident_folders):
        incident_name = folder.name
        images = [
            f for f in sorted(folder.iterdir())
            if f.suffix.lower() in IMAGE_EXTENSIONS
        ]

        print(f"  [{incident_name}] — {len(images)} images found")

        for img_path in images:
            sample = fo.Sample(
                filepath=str(img_path),
                incident=incident_name,          # store incident label
                tags=[incident_name],
            )
            samples.append(sample)

    dataset.add_samples(samples)
    print(f"\n  Total samples loaded: {len(dataset)}")
    return dataset


# ── STEP 2: Near-duplicate detection & removal ───────────────────────────────

def remove_near_duplicates(
    dataset: fo.Dataset,
    threshold: float = UNIQUENESS_THRESHOLD,
    launch_app: bool = True,
) -> fo.DatasetView:
    """
    Compute per-image uniqueness scores using FiftyOne Brain.
    Images with uniqueness < threshold are tagged as near-duplicates.
    Returns a view containing only unique samples.
    """
    print(f"\n{'='*60}")
    print("STEP 2 — Near-duplicate detection")
    print(f"{'='*60}")

    print("  Computing uniqueness scores (this may take a few minutes)...")
    fob.compute_uniqueness(dataset)

    # Tag near-duplicates instead of hard-deleting so you can review first
    dup_view = dataset.match(F("uniqueness") < threshold)
    dup_count = len(dup_view)
    dup_view.tag_samples("near_duplicate")

    print(f"  Uniqueness threshold : {threshold}")
    print(f"  Near-duplicates found: {dup_count}")
    print(f"  Images kept          : {len(dataset) - dup_count}")

    # Optional: open FiftyOne App to visually verify before proceeding
    if launch_app:
        print("\n  Launching FiftyOne App — review tagged near-duplicates,")
        print("  then close the browser tab and press Enter to continue...")
        session = fo.launch_app(dataset.sort_by("uniqueness"))
        input("\n  Press Enter to continue after reviewing...")
        session.close()

    # Return view with near-duplicates excluded
    unique_view = dataset.match(~F("tags").contains("near_duplicate"))
    print(f"\n  Proceeding with {len(unique_view)} unique images.")
    return unique_view


# ── STEP 3: Export all images to single folder ───────────────────────────────

def export_to_single_folder(view: fo.DatasetView, output_dir: str) -> None:
    """
    Copy all unique images to a single output folder.
    """
    print(f"\n{'='*60}")
    print("STEP 3 — Exporting unique images")
    print(f"{'='*60}")

    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    
    total_copied = 0
    for sample in view:
        src = Path(sample.filepath)
        dest = out / src.name
        shutil.copy2(src, dest)
        total_copied += 1

    print(f"  Output directory : {output_dir}")
    print(f"  Total files copied: {total_copied}")



In [3]:

# Step 1 — Load
dataset = load_dataset(DATASET_DIR, DATASET_NAME)

# # Step 2 — Deduplicate
# unique_view = remove_near_duplicates(
#     dataset,
#     threshold=0.75,
#     launch_app=False,   # Set False to skip visual review
# )



STEP 1 — Loading dataset
  [20180812_嘉義縣布袋港嘉明2號擱淺] — 702 images found
  [20190802_嘉義縣布袋港勝利號擱淺_本日油污染較薄] — 26 images found
  [20190803_嘉義縣布袋港勝利號擱淺] — 31 images found
  [20191003_宜蘭縣南方澳大橋斷裂事故油污染事件] — 285 images found
  [20191021_宜蘭縣南方澳大橋斷裂事故油污染事件] — 276 images found
 100% |███████████████| 1320/1320 [363.6ms elapsed, 0s remaining, 3.6K samples/s]     

  Total samples loaded: 1320

STEP 2 — Near-duplicate detection
  Computing uniqueness scores (this may take a few minutes)...
Computing embeddings...
 100% |███████████████| 1320/1320 [1.5m elapsed, 0s remaining, 14.6 samples/s]      
Computing uniqueness...
Uniqueness computation complete
  Uniqueness threshold : 0.5
  Near-duplicates found: 1302
  Images kept          : 18

  Launching FiftyOne App — review tagged near-duplicates,
  then close the browser tab and press Enter to continue...



  Proceeding with 18 unique images.

STEP 3 — Per-incident train / val / test split
  Ratios — train:0.7  val:0.15  test:0.15
  [20180812_嘉義縣布袋港嘉明2號擱淺]  total=7  train=4  val=1  test=2
  [20190803_嘉義縣布袋港勝利號擱淺]  total=7  train=4  val=1  test=2

STEP 4 — Exporting clean dataset
  Output directory : /home/robby/workspace/Oil-Spill-Benz/datasets/processed/dv5_unique
  Total files copied: 18

  Directory structure:
    train/  (12 images)
      20180812_嘉義縣布袋港嘉明2號擱淺/  (4 images)
      20190802_嘉義縣布袋港勝利號擱淺_本日油污染較薄/  (2 images)
      20190803_嘉義縣布袋港勝利號擱淺/  (4 images)
      20191021_宜蘭縣南方澳大橋斷裂事故油污染事件/  (2 images)
    val/  (2 images)
      20180812_嘉義縣布袋港嘉明2號擱淺/  (1 images)
      20190802_嘉義縣布袋港勝利號擱淺_本日油污染較薄/  (0 images)
      20190803_嘉義縣布袋港勝利號擱淺/  (1 images)
      20191021_宜蘭縣南方澳大橋斷裂事故油污染事件/  (0 images)
    test/  (4 images)
      20180812_嘉義縣布袋港嘉明2號擱淺/  (2 images)
      20190802_嘉義縣布袋港勝利號擱淺_本日油污染較薄/  (0 images)
      20190803_嘉義縣布袋港勝利號擱淺/  (2 images)
      20191021_宜蘭縣南方澳大橋斷裂事故油污染事件/  (0 i

In [13]:
dataset

Name:        uav_incidents
Media type:  image
Num samples: 1320
Persistent:  True
Tags:        []
Sample fields:
    id:               fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField
    incident:         fiftyone.core.fields.StringField
    uniqueness:       fiftyone.core.fields.FloatField

In [14]:

# Compute near-duplicates using FiftyOne Brain
print("Computing near-duplicates (this may take a few minutes)...")
index = fob.compute_near_duplicates(dataset, threshold=0.1)

print(index.neighbors_map)

# # Get the duplicate IDs
# duplicate_ids = index.duplicate_ids

# print(f"  Total images        : {len(dataset)}")
# print(f"  Near-duplicates found: {len(duplicate_ids)}")
# print(f"  Unique images       : {len(dataset) - len(duplicate_ids)}")

# # Create view with only unique samples (exclude the duplicates)
# unique_view = dataset.match(~F("id").is_in(duplicate_ids))

# print(f"  Unique view size    : {len(unique_view)}")


Computing near-duplicates (this may take a few minutes)...
Computing embeddings...
 100% |███████████████| 1320/1320 [2.6m elapsed, 0s remaining, 8.2 samples/s]       
Computing duplicate samples...
Duplicates computation complete
{np.str_('69ad22f52b9f3c889510609a'): [(np.str_('69ad22f62b9f3c889510609f'), np.float64(6.9404397607365595)), (np.str_('69ad22f62b9f3c889510609e'), np.float64(6.957087387980326)), (np.str_('69ad22f62b9f3c88951060c6'), np.float64(7.063583128403329)), (np.str_('69ad22f62b9f3c889510609d'), np.float64(7.070968866327344)), (np.str_('69ad22f62b9f3c88951060c7'), np.float64(7.085291956316097)), (np.str_('69ad22f62b9f3c88951060cb'), np.float64(7.110551660739311)), (np.str_('69ad22f62b9f3c889510609c'), np.float64(7.160810125298835)), (np.str_('69ad22f62b9f3c88951060c5'), np.float64(7.211419128004369)), (np.str_('69ad22f62b9f3c88951060ca'), np.float64(7.275651835650055)), (np.str_('69ad22f62b9f3c88951060c8'), np.float64(7.315470286258039)), (np.str_('69ad22f62b9f3c88951

In [15]:
duplicates_view = index.duplicates_view(
    type_field="dup_type",
    id_field="dup_id",
    dist_field="dup_dist",
)

session = fo.launch_app(duplicates_view)

In [9]:

# Launch FiftyOne to visualize unique images
print("Launching FiftyOne App to view unique images...")
session = fo.launch_app(unique_view)
print("✅ FiftyOne App launched. Close the browser tab to continue.")




Launching FiftyOne App to view unique images...


✅ FiftyOne App launched. Close the browser tab to continue.


In [ ]:

# Step 3 — Export to single folder
export_to_single_folder(unique_view, OUTPUT_DIR)

print(f"\n✅ Pipeline complete. Clean dataset saved to: {OUTPUT_DIR}")

